# LangGraph 007 — Parallel Workflows

Fan-out, fan-in, reducers and what parallel actually buys. Needs `langgraph`;
**no API key** — the three "model calls" are sleeps with scripted answers.

| Part | What we check |
|---|---|
| A | three statistics from one innings, in one fan-out |
| B | returning the whole state raises `InvalidUpdateError` |
| C | a reducer merges three one-item lists into `[7, 6, 8]` |
| D | the merge order is **alphabetical by node name** |
| E | waiting branches gain ~3x; computing branches gain ~1x |
| F | one branch fails: the others stay checkpointed |

In [ ]:
import warnings
warnings.filterwarnings("ignore")

## Part A — Fan-out and fan-in

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

class BatsmanState(TypedDict):
    runs: int
    balls: int
    fours: int
    sixes: int
    sr: float                 # strike rate
    bpb: float                # balls per boundary
    boundary_percent: float
    summary: str

# three independent nodes. Each returns ONLY the key it computed.
def calculate_sr(state: BatsmanState) -> dict:
    return {"sr": round(state["runs"] / state["balls"] * 100, 2)}

def calculate_bpb(state: BatsmanState) -> dict:
    return {"bpb": round(state["balls"] / (state["fours"] + state["sixes"]), 2)}

def calculate_boundary_percent(state: BatsmanState) -> dict:
    boundary_runs = state["fours"] * 4 + state["sixes"] * 6
    return {"boundary_percent": round(boundary_runs / state["runs"] * 100, 2)}

def summary(state: BatsmanState) -> dict:
    return {"summary": (f"Strike rate {state['sr']} | "
                        f"Balls per boundary {state['bpb']} | "
                        f"Boundary runs {state['boundary_percent']}%")}

graph = StateGraph(BatsmanState)
graph.add_node("calculate_sr", calculate_sr)
graph.add_node("calculate_bpb", calculate_bpb)
graph.add_node("calculate_boundary_percent", calculate_boundary_percent)
graph.add_node("summary", summary)

# fan out: three edges leaving START. fan in: three edges entering summary.
for node in ("calculate_sr", "calculate_bpb", "calculate_boundary_percent"):
    graph.add_edge(START, node)
    graph.add_edge(node, "summary")
graph.add_edge("summary", END)

app = graph.compile()
out = app.invoke({"runs": 100, "balls": 50, "fours": 6, "sixes": 4,
                  "sr": 0.0, "bpb": 0.0, "boundary_percent": 0.0, "summary": ""})
print(out["sr"], out["bpb"], out["boundary_percent"])
print(out["summary"])

# 200.0 5.0 48.0
# Strike rate 200.0 | Balls per boundary 5.0 | Boundary runs 48.0%

In [ ]:
assert (out["sr"], out["bpb"], out["boundary_percent"]) == (200.0, 5.0, 48.0)
# a second innings through the same compiled graph
other = app.invoke({"runs": 74, "balls": 32, "fours": 5, "sixes": 6,
                    "sr": 0.0, "bpb": 0.0, "boundary_percent": 0.0, "summary": ""})
assert (other["sr"], other["bpb"], other["boundary_percent"]) == (231.25, 2.91, 75.68)
print("three branches, one state, no waiting on each other")

## Part B — The mistake only parallel workflows punish

Each branch below changes the state and hands the whole thing back, so all three
write all eight keys in the same step.

In [ ]:
# The same three nodes, written the way the first workflows in this track were:
# change the state and hand the whole thing back.
def calculate_sr_whole(state: BatsmanState) -> dict:
    state["sr"] = round(state["runs"] / state["balls"] * 100, 2)
    return state                      # <- every key written, by every branch

def calculate_bpb_whole(state: BatsmanState) -> dict:
    state["bpb"] = round(state["balls"] / (state["fours"] + state["sixes"]), 2)
    return state

graph = StateGraph(BatsmanState)
graph.add_node("calculate_sr", calculate_sr_whole)
graph.add_node("calculate_bpb", calculate_bpb_whole)
graph.add_node("summary", summary)
for node in ("calculate_sr", "calculate_bpb"):
    graph.add_edge(START, node)
    graph.add_edge(node, "summary")
graph.add_edge("summary", END)

app = graph.compile()                 # compiles without complaint
try:
    app.invoke({"runs": 100, "balls": 50, "fours": 6, "sixes": 4,
                "sr": 0.0, "bpb": 0.0, "boundary_percent": 0.0, "summary": ""})
except Exception as e:
    print(type(e).__name__)
    print(e)

# InvalidUpdateError
# At key 'runs': Can receive only one value per step. Use an Annotated key to
# handle multiple values.
#
# Both branches wrote every key in the same step. LangGraph has two values for
# one key and no instruction for combining them, so it refuses. Returning only
# the changed key - as in the block above - is the whole fix.

In [ ]:
from langgraph.errors import InvalidUpdateError

# the same two branches, wired one after the other instead of side by side
graph = StateGraph(BatsmanState)
graph.add_node("calculate_sr", calculate_sr_whole)
graph.add_node("calculate_bpb", calculate_bpb_whole)
graph.add_edge(START, "calculate_sr")
graph.add_edge("calculate_sr", "calculate_bpb")
graph.add_edge("calculate_bpb", END)
seq = graph.compile().invoke({"runs": 100, "balls": 50, "fours": 6, "sixes": 4,
                              "sr": 0.0, "bpb": 0.0, "boundary_percent": 0.0,
                              "summary": ""})
assert (seq["sr"], seq["bpb"]) == (200.0, 5.0)
print("in sequence the same nodes are fine - which is why the habit survives")

## Part C — A reducer for a key three branches write

In [ ]:
import operator
from typing import Annotated, TypedDict
from langgraph.graph import StateGraph, START, END

class EssayState(TypedDict):
    essay: str
    language_feedback: str
    analysis_feedback: str
    clarity_feedback: str
    overall_feedback: str
    individual_scores: Annotated[list[int], operator.add]   # three branches write it
    avg_score: float

# Three evaluators, each writing its own feedback key plus a one-item score list.
# The scripted scores stand in for three model calls so this runs with no key.
def evaluate_language(state: EssayState) -> dict:
    return {"language_feedback": "Clear sentences, three idioms used loosely.",
            "individual_scores": [8]}

def evaluate_analysis(state: EssayState) -> dict:
    return {"analysis_feedback": "Supported, but the evidence thins in part three.",
            "individual_scores": [7]}

def evaluate_clarity(state: EssayState) -> dict:
    return {"clarity_feedback": "Well organised; the conclusion restates.",
            "individual_scores": [6]}

def final_evaluation(state: EssayState) -> dict:
    scores = state["individual_scores"]
    return {"overall_feedback": " ".join([state["language_feedback"],
                                          state["analysis_feedback"],
                                          state["clarity_feedback"]]),
            "avg_score": round(sum(scores) / len(scores), 2)}

graph = StateGraph(EssayState)
for name, fn in (("evaluate_language", evaluate_language),
                 ("evaluate_analysis", evaluate_analysis),
                 ("evaluate_clarity", evaluate_clarity)):
    graph.add_node(name, fn)
    graph.add_edge(START, name)
    graph.add_edge(name, "final_evaluation")
graph.add_node("final_evaluation", final_evaluation)
graph.add_edge("final_evaluation", END)

out = graph.compile().invoke({"essay": "India in the age of AI",
                              "language_feedback": "", "analysis_feedback": "",
                              "clarity_feedback": "", "overall_feedback": "",
                              "individual_scores": [], "avg_score": 0.0})
print(out["individual_scores"], out["avg_score"])

# [7, 6, 8] 7.0
#
# Three one-item lists became one three-item list, because `operator.add` on a
# list is concatenation. Note the ORDER: analysis, clarity, language - the node
# names in alphabetical order, not the order they were added to the graph.

In [ ]:
assert sorted(out["individual_scores"]) == [6, 7, 8]
assert out["avg_score"] == 7.0

# the same graph with the reducer taken off the key
from typing import TypedDict

class NoReducer(TypedDict):
    individual_scores: list[int]

graph = StateGraph(NoReducer)
for i, score in enumerate((8, 7, 6)):
    graph.add_node(f"e{i}", (lambda s: lambda st: {"individual_scores": [s]})(score))
    graph.add_edge(START, f"e{i}")
    graph.add_edge(f"e{i}", END)
try:
    graph.compile().invoke({"individual_scores": []})
    raise AssertionError("expected InvalidUpdateError")
except InvalidUpdateError as e:
    print("no reducer ->", str(e).splitlines()[0])

## Part D — What order do merged values arrive in?

The evaluators were added language (8), analysis (7), clarity (6). Guess the
order before running this.

In [ ]:
# Is the merge order the order the nodes were added, or the order they finish?
import operator, time
from typing import Annotated, TypedDict
from langgraph.graph import StateGraph, START, END

class Log(TypedDict):
    log: Annotated[list, operator.add]

def tag(name, delay):
    def node(state):
        time.sleep(delay)             # make one branch finish much later
        return {"log": [name]}
    return node

for names, delays in ((("zulu", "alpha", "mike"), (0.0, 0.0, 0.0)),
                      (("zulu", "alpha", "mike"), (0.0, 0.40, 0.0))):
    graph = StateGraph(Log)
    for name, d in zip(names, delays):
        graph.add_node(name, tag(name, d))
        graph.add_edge(START, name)
        graph.add_edge(name, END)
    print(names, delays, "->", graph.compile().invoke({"log": []})["log"])

# ('zulu', 'alpha', 'mike') (0.0, 0.0, 0.0)  -> ['alpha', 'mike', 'zulu']
# ('zulu', 'alpha', 'mike') (0.0, 0.4, 0.0)  -> ['alpha', 'mike', 'zulu']
#
# Neither. It is alphabetical by node name, and a slow branch does not move.
# So never read a merged list positionally - scores[0] is not "the first
# evaluator". Carry the name with the value if you need to know whose it is.

In [ ]:
# alphabetical, whatever the delays - so positional reads are wrong
assert out["individual_scores"] == [7, 6, 8]
print("scores[0] is", out["individual_scores"][0],
      "- the ANALYSIS score, because evaluate_analysis sorts first")

# carry the name with the value and the question disappears
import operator
from typing import Annotated

class Named(TypedDict):
    scores: Annotated[list, operator.add]

graph = StateGraph(Named)
for name, score in (("language", 8), ("analysis", 7), ("clarity", 6)):
    graph.add_node(name, (lambda nm, sc: lambda s: {"scores": [{nm: sc}]})(name, score))
    graph.add_edge(START, name)
    graph.add_edge(name, END)
merged = graph.compile().invoke({"scores": []})["scores"]
print(merged)
assert {k: v for d in merged for k, v in d.items()}["language"] == 8

## Part E — Fan out the waiting, not the computing

Three branches that sleep 0.30 s each, then three that compute instead. Laptop
timings are noisy, so each figure is the best of three runs.

In [ ]:
import operator, time
from typing import Annotated, TypedDict
from langgraph.graph import StateGraph, START, END

class S(TypedDict):
    log: Annotated[list, operator.add]

def waiter(state):
    time.sleep(0.30)                 # stands in for one model call
    return {"log": ["done"]}

def build(node, parallel, k=3):
    graph = StateGraph(S)
    for i in range(k):
        graph.add_node(f"n{i}", node)
    if parallel:
        for i in range(k):
            graph.add_edge(START, f"n{i}")
            graph.add_edge(f"n{i}", END)
    else:
        graph.add_edge(START, "n0")
        for i in range(k - 1):
            graph.add_edge(f"n{i}", f"n{i + 1}")
        graph.add_edge(f"n{k - 1}", END)
    return graph.compile()

def best_of_three(node, parallel):
    times = []
    for _ in range(3):
        app = build(node, parallel)
        t0 = time.perf_counter()
        app.invoke({"log": []})
        times.append(time.perf_counter() - t0)
    return min(times)

wait_par, wait_seq = best_of_three(waiter, True), best_of_three(waiter, False)
print(f"waiting   parallel {wait_par:.2f} s   sequential {wait_seq:.2f} s   "
      f"{wait_seq / wait_par:.2f}x")
assert wait_seq / wait_par > 2.0        # three waits cost one wait

In [ ]:
def spinner(state):
    total = 0
    for i in range(3_000_000):        # computing, not waiting
        total += i * i
    return {"log": [total]}

cpu_par, cpu_seq = best_of_three(spinner, True), best_of_three(spinner, False)
print(f"computing parallel {cpu_par:.2f} s   sequential {cpu_seq:.2f} s   "
      f"{cpu_seq / cpu_par:.2f}x")
assert cpu_seq / cpu_par < 2.0          # the GIL serialises Python arithmetic
print("fan out model calls, HTTP and database reads - not arithmetic")

## Part F — One branch of three fails

In [ ]:
# One branch of three fails. What happens to the two that succeeded?
from langgraph.checkpoint.memory import InMemorySaver

def evaluate_clarity_broken(state: EssayState) -> dict:
    raise RuntimeError("rate limit from the evaluator")

graph = StateGraph(EssayState)
graph.add_node("evaluate_language", evaluate_language)
graph.add_node("evaluate_analysis", evaluate_analysis)
graph.add_node("evaluate_clarity", evaluate_clarity_broken)
graph.add_node("final_evaluation", final_evaluation)
for name in ("evaluate_language", "evaluate_analysis", "evaluate_clarity"):
    graph.add_edge(START, name)
    graph.add_edge(name, "final_evaluation")
graph.add_edge("final_evaluation", END)

app = graph.compile(checkpointer=InMemorySaver())
config = {"configurable": {"thread_id": "essay-1"}}
blank = {"essay": "India in the age of AI", "language_feedback": "",
         "analysis_feedback": "", "clarity_feedback": "", "overall_feedback": "",
         "individual_scores": [], "avg_score": 0.0}
try:
    app.invoke(blank, config=config)
except RuntimeError as e:
    print("failed:", e)

snapshot = app.get_state(config)
print("scores kept :", snapshot.values["individual_scores"])
print("still to run:", snapshot.next)

# failed: rate limit from the evaluator
# scores kept : [7, 8]
# still to run: ('evaluate_clarity',)
#
# The two good evaluations are saved. Fix the broken branch and `invoke(None,
# config=config)` resumes: only the failed node runs again, and the state ends
# at [7, 6, 8]. You do not pay for the successful calls twice.

In [ ]:
assert sorted(snapshot.values["individual_scores"]) == [7, 8]
assert snapshot.next == ("evaluate_clarity",)

# fix the branch and resume the SAME thread
graph = StateGraph(EssayState)
for name, fn in (("evaluate_language", evaluate_language),
                 ("evaluate_analysis", evaluate_analysis),
                 ("evaluate_clarity", evaluate_clarity)):
    graph.add_node(name, fn)
    graph.add_edge(START, name)
    graph.add_edge(name, "final_evaluation")
graph.add_node("final_evaluation", final_evaluation)
graph.add_edge("final_evaluation", END)
fixed = graph.compile(checkpointer=app.checkpointer)
resumed = fixed.invoke(None, config=config)
print(resumed["individual_scores"], resumed["avg_score"])
assert sorted(resumed["individual_scores"]) == [6, 7, 8]
assert resumed["avg_score"] == 7.0
print("only the failed branch ran again")

## What this notebook established

- Fan-out is several ordinary edges leaving one node; fan-in is several arriving.
- A parallel branch must return **only the keys it changed** — otherwise
  `InvalidUpdateError`, naming the key. The same node is harmless in sequence.
- A key that several branches write needs a reducer:
  `Annotated[list[int], operator.add]`.
- Merged values arrive in **alphabetical order of node name**, not the order
  added and not the order finished. Never index a merged list by position.
- Fan-out on waiting gained about **3x**; on computing, about **1x**.
- With a checkpointer, a failed branch leaves the successful ones saved, and
  resuming re-runs only the failure.

## Exercises

1. Add a fourth statistic (dot-ball percentage needs a `dots` input) and check
   that nothing else in the graph has to change.
2. Make two branches write `summary` instead of their own keys. Fix it twice:
   once with a reducer, once by renaming the keys. Which would you ship?
3. Rename the three evaluators `a_language`, `b_analysis`, `c_clarity` and
   predict the merged list before you run it.
4. Give the failing branch a `RetryPolicy` (lesson 015 covers it) and see
   whether the resume is still needed.